## Data Preparation

Tahap ini bertujuan untuk menyiapkan dataset yang akan digunakan pada proses analisis dan pembangunan model machine learning. Proses yang dilakukan meliputi pembacaan data NASA POWER, pemeriksaan kualitas data, harmonisasi spasial menggunakan interpolasi bilinear, integrasi seluruh variabel menjadi satu dataset master, validasi hasil pengolahan, serta penyimpanan dataset dalam format yang siap digunakan pada tahap selanjutnya.

#### Konfigurasi Awal

Tahap ini menyiapkan lingkungan kerja yang digunakan selama proses data preparation. Konfigurasi meliputi import library, penentuan lokasi penyimpanan data, pendefinisian variabel NASA POWER yang digunakan dalam penelitian, serta parameter umum yang menjadi acuan selama proses pengolahan data.

In [39]:
# Import library dan konfigurasi tampilan pandas

import io
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


# Direktori data

DIR_RAW = Path("../data/raw")
DIR_PROCESSED = Path("../data/processed")


# Konfigurasi variabel NASA POWER berdasarkan sumber produk data

VAR_IRRADIANCE = {
"GHI": ("GHI", "ALLSKY_SFC_SW_DWN"),    # TARGET model
"DHI": ("DHI", "ALLSKY_SFC_SW_DIFF"),   # komponen diffuse
"DNI": ("DNI", "ALLSKY_SFC_SW_DNI"),    # komponen direct normal
}

VAR_METEO = {
"TEMP":   ("Temperature",   "T2M"),          # suhu 2 meter
"RH":     ("Humidity",      "RH2M"),         # kelembapan relatif 2 meter
"PRECIP": ("Precipitation", "PRECTOTCORR"),  # presipitasi terkoreksi
}

VAR_ALL = {**VAR_IRRADIANCE, **VAR_METEO}


# Parameter periode dan penanganan data

VAR_YEAR = [2021, 2022, 2023, 2024, 2025]
VAR_NASA_POWER_NAN = -999
DESIMAL_KOORDINAT = 4


# Ringkasan konfigurasi untuk verifikasi cepat

print(f"DIR_MENTAH : {DIR_RAW.resolve()}")
print(f"DIR_OLAHAN : {DIR_PROCESSED.resolve()}")
print(f"Tahun      : {VAR_YEAR}")
print(f"Variabel   : {list(VAR_ALL)}")

DIR_MENTAH : C:\Users\Z00588HW\OneDrive - Siemens AG\tugas-akhir-angga\ml\data\raw
DIR_OLAHAN : C:\Users\Z00588HW\OneDrive - Siemens AG\tugas-akhir-angga\ml\data\processed
Tahun      : [2021, 2022, 2023, 2024, 2025]
Variabel   : ['GHI', 'DHI', 'DNI', 'TEMP', 'RH', 'PRECIP']


#### Memuat Data NASA POWER

Tahap ini bertujuan untuk membaca seluruh dataset NASA POWER yang digunakan dalam penelitian. Setiap file dibaca dengan mengabaikan bagian metadata sehingga hanya data observasi yang diproses. Selanjutnya, data dari setiap tahun digabungkan menjadi satu dataset untuk masing-masing variabel.

In [40]:
# Fungsi untuk membaca file CSV dari folder data raw

def load_csv(file_path: Path) -> pd.DataFrame:
    # Baca seluruh isi file dan cari akhir metadata header
    with io.open(file_path, mode="r", encoding="utf-8") as f:
        content = f.readlines()

    idx_header_end = next(
        (i for i, line in enumerate(content) if line.strip().startswith("-END HEADER-")),
        None
    )

    # Validasi keberadaan penutup header
    if idx_header_end is None:
        raise ValueError(f"File {file_path} tidak memiliki baris penutup header '-END HEADER-'")

    # Parse bagian data setelah metadata header menjadi DataFrame
    data_str = "".join(content[idx_header_end + 1:])
    return pd.read_csv(io.StringIO(data_str))


# Fungsi untuk memuat seluruh data tahunan dari satu variabel NASA POWER

def load_variable(short_name: str) -> pd.DataFrame:
    # Tentukan prefiks file dan nama parameter berdasarkan variabel
    file_prefix, expected_param = VAR_ALL[short_name]
    yearly_data = []

    # Baca dan proses data untuk setiap tahun
    for year in VAR_YEAR:
        file_path = DIR_RAW / f"{file_prefix}_Daily_{year}.csv"

        if not file_path.exists():
            raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

        raw_df = load_csv(file_path)

        # Validasi dan standarisasi nama kolom
        value_column = raw_df.columns[-1]

        if value_column != expected_param:
            print(
                f"[!] {file_path.name}: kolom '{value_column}' "
                f"tidak sesuai dengan '{expected_param}'"
            )

        df = raw_df.rename(
            columns={
                value_column: short_name,
                "LAT": "lat",
                "LON": "lon"
            }
        )

        # Bentuk tanggal, tangani nilai hilang, dan standarisasi koordinat
        df["date"] = pd.to_datetime(
            dict(
                year=df["YEAR"],
                month=df["MO"],
                day=df["DY"]
            )
        )

        df[short_name] = df[short_name].replace(VAR_NASA_POWER_NAN, np.nan)

        df["lat"] = df["lat"].round(DESIMAL_KOORDINAT)
        df["lon"] = df["lon"].round(DESIMAL_KOORDINAT)

        # Simpan hanya kolom yang diperlukan
        yearly_data.append(df[["lat", "lon", "date", short_name]])
        
    # Gabungkan seluruh data tahunan dan urutkan berdasarkan lokasi serta tanggal
    result = pd.concat(yearly_data, ignore_index=True)
    result = result.sort_values(["lat", "lon", "date"]).reset_index(drop=True)

    return result


# Fungsi untuk memperoleh daftar koordinat unik

def get_grid(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    # Mengembalikan latitude dan longitude unik dalam urutan menaik
    return np.sort(df["lat"].unique()), np.sort(df["lon"].unique())


print("Fungsi loader siap.")

Fungsi loader siap.


#### Pemeriksaan Kualitas Data

Setelah seluruh data berhasil dimuat, dilakukan pemeriksaan kualitas data untuk memastikan setiap variabel memiliki cakupan data yang sesuai. Pemeriksaan meliputi jumlah baris, jumlah lokasi, rentang tanggal, jumlah missing value, serta rentang nilai setiap variabel.

In [41]:
# Muat seluruh variabel NASA POWER ke dalam dictionary

data_all = {}

for var_name in VAR_ALL:
    data_all[var_name] = load_variable(var_name)


# Lakukan quality check pada seluruh variabel

qc_result = []

for var_name, df in data_all.items():

    # Ambil informasi grid dan nilai variabel
    lat, lon = get_grid(df)
    value = df[var_name]

    # Simpan ringkasan hasil quality check
    qc_result.append({
        "variabel": var_name,
        "sumber": "CERES" if var_name in VAR_IRRADIANCE else "MERRA-2",
        "n_baris": len(df),
        "n_lokasi": len(lat) * len(lon),
        "tgl_min": df["date"].min().date(),
        "tgl_max": df["date"].max().date(),
        "n_missing": int(value.isna().sum()),
        "%_missing": round(value.isna().mean() * 100, 4),
        "nilai_min": round(float(value.min()), 3),
        "nilai_max": round(float(value.max()), 3),
    })


# Ubah hasil quality check menjadi DataFrame untuk ditampilkan

df_qc = pd.DataFrame(qc_result)
df_qc = df_qc.set_index("variabel")

df_qc

,sumber,n_baris,n_lokasi,tgl_min,tgl_max,n_missing,%_missing,nilai_min,nilai_max
variabel,,,,,,,,,
GHI,CERES,73040,40,2021-01-01,2025-12-31,0,0.0000,0.355,7.764
DHI,CERES,73040,40,2021-01-01,2025-12-31,40,0.0548,0.268,3.994
DNI,CERES,73040,40,2021-01-01,2025-12-31,40,0.0548,0.000,9.474
TEMP,MERRA-2,219120,120,2021-01-01,2025-12-31,0,0.0000,18.580,32.260
RH,MERRA-2,219120,120,2021-01-01,2025-12-31,0,0.0000,45.800,97.710
PRECIP,MERRA-2,219120,120,2021-01-01,2025-12-31,0,0.0000,0.000,318.680


#### Verifikasi Grid Koordinat

Tahap ini memverifikasi bahwa seluruh variabel dalam kelompok irradiance maupun meteorologi memiliki grid koordinat yang konsisten. Selain itu, dilakukan identifikasi titik koordinat CERES yang berada di luar cakupan grid MERRA-2 sebagai dasar proses harmonisasi spasial.

In [42]:
# Ambil grid koordinat untuk masing-masing kelompok data

grid_irradiance = {
    var_name: get_grid(data_all[var_name])
    for var_name in VAR_IRRADIANCE
}

grid_meteo = {
    var_name: get_grid(data_all[var_name])
    for var_name in VAR_METEO
}


# Fungsi untuk memeriksa konsistensi grid antar variabel

def check_grid_consistency(grid_group: dict) -> bool:
    ref_lat, ref_lon = next(iter(grid_group.values()))

    return all(
        np.array_equal(ref_lat, lat) and np.array_equal(ref_lon, lon)
        for lat, lon in grid_group.values()
    )


# Validasi konsistensi grid pada masing-masing kelompok data

assert check_grid_consistency(grid_irradiance), (
    "Grid CERES tidak konsisten antar variabel."
)

assert check_grid_consistency(grid_meteo), (
    "Grid MERRA-2 tidak konsisten antar variabel."
)


# Tetapkan grid CERES sebagai grid target dan MERRA-2 sebagai grid meteorologi

CERES_LAT, CERES_LON = get_grid(data_all["GHI"])
MERRA_LAT, MERRA_LON = get_grid(data_all["TEMP"])


# Tampilkan informasi grid CERES

print("Grid CERES")
print("Latitude :", CERES_LAT)
print("Longitude:", CERES_LON)
print("Jumlah titik:", len(CERES_LAT) * len(CERES_LON))


# Tampilkan informasi grid MERRA-2

print("\nGrid MERRA-2")
print("Latitude :", MERRA_LAT)
print("Longitude:", MERRA_LON)
print("Jumlah titik:", len(MERRA_LAT) * len(MERRA_LON))


# Identifikasi koordinat CERES yang berada di luar cakupan grid MERRA-2

lon_outside = CERES_LON[
    (CERES_LON < MERRA_LON.min()) |
    (CERES_LON > MERRA_LON.max())
]

lat_outside = CERES_LAT[
    (CERES_LAT < MERRA_LAT.min()) |
    (CERES_LAT > MERRA_LAT.max())
]


# Tampilkan koordinat di luar cakupan MERRA-2

print("\nLongitude di luar cakupan MERRA-2:", lon_outside)
print("Latitude di luar cakupan MERRA-2 :", lat_outside)

Grid CERES
Latitude : [-8.5 -7.5 -6.5 -5.5]
Longitude: [105.5 106.5 107.5 108.5 109.5 110.5 111.5 112.5 113.5 114.5]
Jumlah titik: 40

Grid MERRA-2
Latitude : [-9.  -8.5 -8.  -7.5 -7.  -6.5 -6.  -5.5]
Longitude: [105.625 106.25  106.875 107.5   108.125 108.75  109.375 110.    110.625
 111.25  111.875 112.5   113.125 113.75  114.375]
Jumlah titik: 120

Longitude di luar cakupan MERRA-2: [105.5 114.5]
Latitude di luar cakupan MERRA-2 : []


#### Menggabungkan Variabel Irradiance

Variabel irradiance yang berasal dari produk CERES SYN1deg digabungkan menjadi satu dataset berdasarkan koordinat dan tanggal. Proses ini menghasilkan dataset irradiance yang akan digunakan sebagai acuan pada tahap integrasi selanjutnya.

In [43]:
# Gabungkan seluruh variabel irradiance berdasarkan lokasi dan tanggal

df_irradiance = data_all["GHI"]

for var_name in ["DHI", "DNI"]:
    df_irradiance = df_irradiance.merge(
        data_all[var_name],
        on=["lat", "lon", "date"],
        how="outer"
    )


# Urutkan data dan hitung ringkasan jumlah lokasi serta hari

df_irradiance = (
    df_irradiance
    .sort_values(["lat", "lon", "date"])
    .reset_index(drop=True)
)

total_location = (
    df_irradiance[["lat", "lon"]]
    .drop_duplicates()
    .shape[0]
)

total_day = df_irradiance["date"].nunique()


# Tampilkan ringkasan data irradiance

print(f"Jumlah baris : {len(df_irradiance):,}")
print(f"Jumlah lokasi: {total_location}")
print(
    f"Rentang data : {df_irradiance['date'].min().date()} "
    f"sampai {df_irradiance['date'].max().date()}"
)
print(f"Jumlah hari  : {total_day}")
print(f"Ekspektasi   : {total_location * total_day:,} baris")


# Tampilkan lima data pertama

df_irradiance.head()

Jumlah baris : 73,040
Jumlah lokasi: 40
Rentang data : 2021-01-01 sampai 2025-12-31
Jumlah hari  : 1826
Ekspektasi   : 73,040 baris


,lat,lon,date,GHI,DHI,DNI
0,-8.5,105.5,2021-01-01,3.0562,2.2087,0.1589
1,-8.5,105.5,2021-01-02,4.3577,3.1548,0.6300
2,-8.5,105.5,2021-01-03,6.7411,2.6938,5.6172
3,-8.5,105.5,2021-01-04,5.0578,3.0982,2.5937
4,-8.5,105.5,2021-01-05,5.3494,3.5186,2.3918


#### Menyusun Rencana Interpolasi Bilinear

Tahap ini menyusun rencana interpolasi bilinear untuk setiap titik pada grid CERES berdasarkan grid MERRA-2. Rencana yang dihasilkan berupa pasangan titik pengapit, bobot interpolasi, serta informasi apakah suatu titik memerlukan extrapolation karena berada di luar cakupan grid sumber.

In [44]:
from dataclasses import dataclass
import numpy as np
 
 
# Menyimpan informasi interpolasi untuk setiap titik target
 
@dataclass
class BilinearPlan:
    lat: float
    lon: float
    corners: list
    weights: np.ndarray
    clamped: bool  # True jika koordinat target di luar cakupan grid sumber dan dibatasi (di-clamp) ke tepi grid
 
 
# Fungsi untuk mencari dua titik grid yang mengapit koordinat target
 
def find_surrounding_index(
    coordinate: np.ndarray,
    value: float
) -> tuple[int, int, float, bool]:
 
    # Batasi koordinat target pada rentang grid sumber
    min_value = coordinate[0]
    max_value = coordinate[-1]
 
    is_clamped = value < min_value or value > max_value
    value = min(max(value, min_value), max_value)
 
    # Tentukan indeks titik grid yang mengapit koordinat target
    if value >= max_value:
        idx0 = len(coordinate) - 2
        idx1 = len(coordinate) - 1
    else:
        idx1 = int(np.searchsorted(coordinate, value, side="right"))
        idx0 = idx1 - 1
 
    # Hitung bobot interpolasi pada sumbu koordinat
    weight = (
        (value - coordinate[idx0]) /
        (coordinate[idx1] - coordinate[idx0])
    )
 
    return idx0, idx1, weight, is_clamped
 
 
# Fungsi untuk menyusun rencana interpolasi bilinear
 
def build_bilinear_plan(
    target_lat,
    target_lon,
    source_lat,
    source_lon
) -> list:
 
    plans = []
 
    # Susun rencana interpolasi untuk setiap titik target
    for lat in target_lat:
        for lon in target_lon:
 
            lat0, lat1, weight_y, clamp_lat = (
                find_surrounding_index(source_lat, lat)
            )
 
            lon0, lon1, weight_x, clamp_lon = (
                find_surrounding_index(source_lon, lon)
            )
 
            # Hitung bobot untuk keempat titik sudut grid
            weights = np.array([
                (1 - weight_y) * (1 - weight_x),
                (1 - weight_y) * weight_x,
                weight_y * (1 - weight_x),
                weight_y * weight_x,
            ])
 
            # Simpan titik sudut, bobot, dan status clamped.
            # PENTING: titik yang clamped di sini TIDAK diekstrapolasi
            # secara linear. Nilainya di-clamp persis ke tepi grid sumber,
            # sehingga weight menjadi tepat 0 atau 1 dan hasil interpolasi
            # = nilai grid tepi terdekat. Ini "clamping", bukan extrapolation.
            plans.append(
                BilinearPlan(
                    lat=float(lat),
                    lon=float(lon),
                    corners=[
                        (lat0, lon0),
                        (lat0, lon1),
                        (lat1, lon0),
                        (lat1, lon1),
                    ],
                    weights=weights,
                    clamped=clamp_lat or clamp_lon,
                )
            )
 
    return plans
 
 
# Susun rencana interpolasi dari grid MERRA-2 ke grid CERES
 
bilinear_plan = build_bilinear_plan(
    CERES_LAT,
    CERES_LON,
    MERRA_LAT,
    MERRA_LON
)
 
 
# Hitung dan tampilkan jumlah titik yang nilainya di-clamp ke tepi grid sumber
 
total_clamped = sum(
    point.clamped
    for point in bilinear_plan
)
 
print(f"Jumlah titik target : {len(bilinear_plan)}")
print(f"Titik di-clamp      : {total_clamped} (nilai dibatasi ke tepi grid, bukan diekstrapolasi)")

Jumlah titik target : 40
Titik di-clamp      : 8 (nilai dibatasi ke tepi grid, bukan diekstrapolasi)


### Harmonisasi Data Meteorologi

Harmonisasi spasial dilakukan dengan menerapkan interpolasi bilinear pada seluruh variabel meteorologi dari grid MERRA-2 ke grid CERES. Hasilnya berupa dataset meteorologi yang telah memiliki resolusi spasial yang sama dengan data irradiance.

In [45]:
# Fungsi untuk menyusun data menjadi kubus (tanggal × latitude × longitude)

def build_data_cube(
    df: pd.DataFrame,
    variable: str,
    source_lat,
    source_lon
):

    # Bentuk indeks tanggal dan koordinat untuk penyusunan kubus
    dates = np.sort(df["date"].unique())

    date_index = {date: idx for idx, date in enumerate(dates)}
    lat_index = {lat: idx for idx, lat in enumerate(source_lat)}
    lon_index = {lon: idx for idx, lon in enumerate(source_lon)}

    # Inisialisasi kubus data dengan nilai NaN
    cube = np.full(
        (len(dates), len(source_lat), len(source_lon)),
        np.nan,
        dtype="float64"
    )

    # Petakan setiap data ke posisi kubus dan masukkan nilainya
    date_pos = df["date"].map(date_index).to_numpy()
    lat_pos = df["lat"].map(lat_index).to_numpy()
    lon_pos = df["lon"].map(lon_index).to_numpy()

    cube[date_pos, lat_pos, lon_pos] = df[variable].to_numpy()

    return cube, dates


# Fungsi untuk melakukan interpolasi bilinear pada data meteorologi

def interpolate_meteo(
    df: pd.DataFrame,
    variable: str,
    bilinear_plan,
    source_lat,
    source_lon
) -> pd.DataFrame:

    cube, dates = build_data_cube(
        df,
        variable,
        source_lat,
        source_lon
    )

    result = []

    for point in bilinear_plan:

        corner_value = np.stack(
            [cube[:, i, j] for i, j in point.corners],
            axis=1
        )

        weight = point.weights[None, :]
        valid = ~np.isnan(corner_value)
        effective_weight = weight * valid

        total_weight = effective_weight.sum(axis=1)
        numerator = np.nansum(
            corner_value * effective_weight,
            axis=1
        )

        interpolated = np.where(
            total_weight > 0,
            numerator / np.where(total_weight > 0, total_weight, 1),
            np.nan
        )

        result.append(
            pd.DataFrame({
                "lat": point.lat,
                "lon": point.lon,
                "date": dates,
                variable: interpolated,
                "meteo_clamped": point.clamped,
            })
        )

    return pd.concat(result, ignore_index=True)


df_meteo = None

for var_name in VAR_METEO:

    result = interpolate_meteo(
        data_all[var_name],
        var_name,
        bilinear_plan,
        MERRA_LAT,
        MERRA_LON
    )

    if df_meteo is None:
        df_meteo = result
    else:
        df_meteo = df_meteo.merge(
            result.drop(columns="meteo_clamped"),
            on=["lat", "lon", "date"],
            how="outer"
        )

    print(f"{var_name} selesai ({len(result):,} baris)")


# Urutkan dan tampilkan ringkasan hasil harmonisasi

df_meteo = (
    df_meteo
    .sort_values(["lat", "lon", "date"])
    .reset_index(drop=True)
)

print(f"Shape data: {df_meteo.shape}")

df_meteo.head()

TEMP selesai (73,040 baris)
RH selesai (73,040 baris)
PRECIP selesai (73,040 baris)
Shape data: (73040, 7)


,lat,lon,date,TEMP,meteo_clamped,RH,PRECIP
0,-8.5,105.5,2021-01-01,26.71,True,88.29,39.22
1,-8.5,105.5,2021-01-02,26.85,True,83.84,8.22
2,-8.5,105.5,2021-01-03,27.47,True,83.02,7.17
3,-8.5,105.5,2021-01-04,27.47,True,85.52,8.71
4,-8.5,105.5,2021-01-05,27.85,True,83.17,4.01


### Integrasi Dataset

Dataset irradiance dan dataset meteorologi yang telah diharmonisasi kemudian digabungkan berdasarkan koordinat dan tanggal. Hasil proses ini adalah dataset master yang berisi seluruh variabel penelitian pada grid spasial yang seragam.

In [46]:
# Gabungkan data irradiance dan meteorologi berdasarkan lokasi dan tanggal

df_master = df_irradiance.merge(
    df_meteo,
    on=["lat", "lon", "date"],
    how="inner"
)


# Buat ID unik dan tetapkan struktur kolom dataset

df_master["location_id"] = (
    df_master["lat"].map(lambda x: f"{x:.4f}")
    + "_"
    + df_master["lon"].map(lambda x: f"{x:.4f}")
)

column_order = [
    "location_id",
    "lat",
    "lon",
    "date",
    "GHI",
    "DHI",
    "DNI",
    "TEMP",
    "RH",
    "PRECIP",
    "meteo_clamped",
]


# Susun dan urutkan dataset berdasarkan lokasi dan tanggal

df_master = (
    df_master[column_order]
    .sort_values(["location_id", "date"])
    .reset_index(drop=True)
)


# Tampilkan ringkasan dan sampel dataset

print(f"Shape data     : {df_master.shape}")
print(f"Jumlah lokasi  : {df_master['location_id'].nunique()}")
print(
    f"Rentang tanggal: "
    f"{df_master['date'].min().date()} "
    f"sampai "
    f"{df_master['date'].max().date()}"
)

df_master.head(10)

Shape data     : (73040, 11)
Jumlah lokasi  : 40
Rentang tanggal: 2021-01-01 sampai 2025-12-31


,location_id,lat,lon,date,GHI,DHI,DNI,TEMP,RH,PRECIP,meteo_clamped
0,-5.5000_105.5000,-5.5,105.5,2021-01-01,5.9597,2.6189,3.7409,27.00,82.81,5.64,True
1,-5.5000_105.5000,-5.5,105.5,2021-01-02,5.7943,3.1807,2.7691,26.67,84.42,10.10,True
2,-5.5000_105.5000,-5.5,105.5,2021-01-03,4.3733,2.9808,0.7447,27.01,82.85,13.56,True
3,-5.5000_105.5000,-5.5,105.5,2021-01-04,5.3333,3.0636,2.5296,26.97,82.29,10.85,True
4,-5.5000_105.5000,-5.5,105.5,2021-01-05,3.0929,2.2714,0.1898,26.73,87.08,3.04,True
5,-5.5000_105.5000,-5.5,105.5,2021-01-06,2.8166,1.8007,0.6972,26.92,88.64,13.37,True
6,-5.5000_105.5000,-5.5,105.5,2021-01-07,4.4002,3.1222,0.6010,26.87,86.18,11.87,True
7,-5.5000_105.5000,-5.5,105.5,2021-01-08,3.0233,2.2248,0.1541,26.86,86.06,4.93,True
8,-5.5000_105.5000,-5.5,105.5,2021-01-09,3.1553,2.2526,0.3958,26.69,87.61,12.27,True
9,-5.5000_105.5000,-5.5,105.5,2021-01-10,4.5350,3.3554,0.5071,26.87,84.07,2.94,True


### Validasi Dataset

Tahap validasi dilakukan untuk memastikan struktur dataset telah sesuai dengan kebutuhan penelitian. Pemeriksaan meliputi duplikasi data, kelengkapan rentang waktu, jumlah observasi pada setiap lokasi, serta validitas rentang nilai masing-masing variabel.

In [47]:
# Hitung jumlah hari dan lokasi pada periode penelitian

total_day = pd.date_range(
    "2021-01-01",
    "2025-12-31",
    freq="D"
).size

total_location = df_master["location_id"].nunique()


# Validasi struktur, kelengkapan, dan rentang nilai dataset

validation = {
    "Tidak ada data duplikat":
        df_master.duplicated(["location_id", "date"]).sum() == 0,

    "Jumlah hari setiap lokasi sama":
        df_master.groupby("location_id")["date"].nunique().nunique() == 1,

    "Jumlah hari sesuai periode penelitian":
        int(
            df_master.groupby("location_id")["date"]
            .nunique()
            .iloc[0]
        ) == total_day,

    "Jumlah baris sesuai":
        len(df_master) == total_location * total_day,

    "RH berada pada rentang 0–100":
        df_master["RH"].dropna().between(0, 100).all(),

    "GHI, DHI, dan DNI bernilai positif":
        (df_master[["GHI", "DHI", "DNI"]].fillna(0) >= 0).all().all(),

    "PRECIP bernilai positif":
        (df_master["PRECIP"].fillna(0) >= 0).all(),
}


# Tampilkan hasil validasi

for name, status in validation.items():
    print(f"[{'OK' if status else 'FAIL'}] {name}")


# Hentikan proses apabila terdapat validasi yang gagal

assert all(validation.values()), (
    "Validasi dataset gagal. Periksa proses sebelumnya."
)

print("\nSeluruh validasi dataset berhasil.")

[OK] Tidak ada data duplikat
[OK] Jumlah hari setiap lokasi sama
[OK] Jumlah hari sesuai periode penelitian
[OK] Jumlah baris sesuai
[OK] RH berada pada rentang 0–100
[OK] GHI, DHI, dan DNI bernilai positif
[OK] PRECIP bernilai positif

Seluruh validasi dataset berhasil.


### Ringkasan Kualitas Dataset

Tahap ini menyajikan ringkasan kualitas dataset hasil integrasi. Informasi yang ditampilkan meliputi jumlah missing value, statistik dasar setiap variabel, serta daftar lokasi yang menggunakan hasil extrapolation pada proses harmonisasi spasial.

In [50]:
# Tentukan variabel yang akan dianalisis dan hitung statistik deskriptif

value_columns = [
    "GHI",
    "DHI",
    "DNI",
    "TEMP",
    "RH",
    "PRECIP",
]

df_summary = pd.DataFrame({
    "n_missing": df_master[value_columns].isna().sum(),
    "%_missing": (
        df_master[value_columns]
        .isna()
        .mean() * 100
    ).round(4),
    "min": df_master[value_columns].min().round(3),
    "max": df_master[value_columns].max().round(3),
    "mean": df_master[value_columns].mean().round(3),
})


# Tampilkan ringkasan statistik

display(df_summary)


# Identifikasi lokasi yang menggunakan hasil extrapolation

df_extrapolated = (
    df_master.loc[
        df_master["meteo_clamped"],
        ["lat", "lon"]
    ]
    .drop_duplicates()
    .sort_values(["lat", "lon"])
    .reset_index(drop=True)
)


# Tampilkan jumlah dan daftar lokasi extrapolation

print(
    f"Jumlah lokasi clamping: "
    f"{len(df_extrapolated)} "
    f"dari "
    f"{df_master['location_id'].nunique()}"
)

display(df_extrapolated)

,n_missing,%_missing,min,max,mean
GHI,0,0.0000,0.355,7.764,5.084
DHI,40,0.0548,0.268,3.994,2.372
DNI,40,0.0548,0.000,9.474,3.329
TEMP,0,0.0000,21.030,31.880,27.334
RH,0,0.0000,45.800,96.188,81.898
PRECIP,0,0.0000,0.000,318.680,6.557


Jumlah lokasi clamping: 8 dari 40


,lat,lon
0,-8.5,105.5
1,-8.5,114.5
2,-7.5,105.5
3,-7.5,114.5
4,-6.5,105.5
5,-6.5,114.5
6,-5.5,105.5
7,-5.5,114.5


In [51]:
# Identifikasi outlier menggunakan metode IQR

for col in ['GHI', 'DNI', 'DHI', 'TEMP', 'RH', 'PRECIP']:
    Q1 = df_master[col].quantile(0.25)
    Q3 = df_master[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_master[
        (df_master[col] < (Q1 - 1.5 * IQR)) |
        (df_master[col] > (Q3 + 1.5 * IQR))
    ].shape

    print(f"{col}: {outliers} baris outliers")

GHI: (2303, 11) baris outliers
DNI: (0, 11) baris outliers
DHI: (530, 11) baris outliers
TEMP: (1300, 11) baris outliers
RH: (1617, 11) baris outliers
PRECIP: (5268, 11) baris outliers


### Menyimpan Dataset

Dataset master yang telah lolos proses validasi disimpan ke dalam format CSV dan Parquet. Dataset ini menjadi keluaran akhir dari tahap data preparation dan selanjutnya digunakan pada proses exploratory data analysis dan pembangunan model machine learning.

In [52]:
# Tentukan lokasi output dan simpan dataset dalam format CSV

csv_path = DIR_PROCESSED / "master_dataset.csv"
parquet_path = DIR_PROCESSED / "master_dataset.parquet"

df_master.to_csv(csv_path, index=False)

print(
    f"CSV berhasil disimpan "
    f"({csv_path.stat().st_size / 1e6:.2f} MB)"
)


# Simpan dataset dalam format Parquet jika dependensi tersedia

try:
    df_master.to_parquet(parquet_path, index=False)

    print(
        f"Parquet berhasil disimpan "
        f"({parquet_path.stat().st_size / 1e6:.2f} MB)"
    )

except Exception as err:
    print(f"Gagal menyimpan Parquet: {err}")
    print("Install pyarrow untuk mengaktifkan format Parquet.")


# Tampilkan ringkasan akhir proses data preparation

print("\nDATA PREPARATION SELESAI")
print(f"Jumlah baris : {len(df_master):,}")
print(f"Jumlah kolom : {df_master.shape[1]}")
print(f"Jumlah lokasi: {df_master['location_id'].nunique()}")
print(
    f"Rentang data : "
    f"{df_master['date'].min().date()} "
    f"sampai "
    f"{df_master['date'].max().date()}"
)

CSV berhasil disimpan (6.70 MB)
Parquet berhasil disimpan (1.43 MB)

DATA PREPARATION SELESAI
Jumlah baris : 73,040
Jumlah kolom : 11
Jumlah lokasi: 40
Rentang data : 2021-01-01 sampai 2025-12-31
